In [ ]:
!pip install torchbnn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchbnn as bnn
import numpy as np
from scipy.stats import norm
from sklearn.calibration import calibration_curve
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pickle
import os
import copy
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score
from sklearn.calibration import calibration_curve
from scipy.stats import wilcoxon
from copy import deepcopy
from torch.utils.data import random_split

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------- Helper: Sign Entropy for DropConnect ----------------

def calculate_entropy_dist(mu, sigma):
    # Standard normal distribution
    standard_normal = torch.distributions.Normal(loc=0.0, scale=1.0)

    # Compute z = (0 - mu) / sigma
    z = -mu / (sigma + 1e-8)  # tiny epsilon ONLY for division safety

    # CDF of z under standard normal
    p_negative = standard_normal.cdf(z)
    p_positive = 1 - p_negative

    # Create entropy tensor of same shape
    entropy = torch.zeros_like(mu)

    # Mask where p_positive is between (1e-10, 1-1e-10)
    nondeterministic_mask = (p_positive > 1e-10) & (p_positive < 1 - 1e-10)

    # Compute entropy only where it matters
    p_pos = p_positive[nondeterministic_mask]
    p_neg = p_negative[nondeterministic_mask]

    entropy[nondeterministic_mask] = -(p_pos * torch.log2(p_pos) + p_neg * torch.log2(p_neg))

    return entropy


def conv_block(in_channels, out_channels, pool=False):
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    ]
    if pool:
        layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)


class BayesianVGG16_FC(nn.Module):
    def __init__(
        self,
        drop_type=None,
        drop_prob=0.3,
        alpha=50,
        num_classes=10,
        in_channels=1
    ):
        super().__init__()

        self.backbone = models.vgg16(weights=None)

        # For Fashion-MNIST / MNIST use in_channels=1
        # For CIFAR-10 use in_channels=3
        self.backbone.features[0] = nn.Conv2d(
            in_channels,
            64,
            kernel_size=3,
            stride=1,
            padding=1
        )

        # Important for 32x32 images.
        # Standard VGG classifier expects 7x7 features.
        # We make it compact: 512 x 1 x 1 -> 512.
        self.backbone.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.feature_dim = 512

        self.drop_type = drop_type
        self.drop_prob = drop_prob
        self.alpha = alpha

        if self.drop_type == "dropout":
            self.dropout_layer = nn.Dropout(p=self.drop_prob)

        # Bayesian final classifier
        self.classifier = bnn.BayesLinear(
            prior_mu=0,
            prior_sigma=0.1,
            in_features=self.feature_dim,
            out_features=num_classes
        )

    def forward_features(self, x):
        x = self.backbone.features(x)
        x = self.backbone.avgpool(x)
        x = torch.flatten(x, 1)
        return x

    def forward(self, x, current_epoch=None, total_epochs=None):
        x = self.forward_features(x)

        fc = self.classifier

        if self.training:
            if self.drop_type == "dropout":
                # Dropout acts on activation vector before final classifier
                x = self.dropout_layer(x)
                logits = fc(x)

            elif self.drop_type == "dropconnect":
                masked_mu, masked_log_sigma = self.random_dropconnect(
                    fc.weight_mu,
                    fc.weight_log_sigma
                )
                logits = F.linear(
                    x,
                    masked_mu,
                    fc.bias_mu if fc.bias else None
                )

            elif self.drop_type == "signentropy":
                fc.weight_mu.data, fc.weight_log_sigma.data = self.signentropy_dropconnect_train(
                    fc.weight_mu.data,
                    fc.weight_log_sigma.data,
                    current_epoch,
                    total_epochs
                )
                logits = fc(x)

            else:
                logits = fc(x)

        else:
            if self.drop_type == "dropconnect":
                masked_mu = fc.weight_mu * (1 - self.drop_prob)
                logits = F.linear(
                    x,
                    masked_mu,
                    fc.bias_mu if fc.bias else None
                )

            elif self.drop_type == "signentropy":
                logits = F.linear(
                    x,
                    fc.weight_mu,
                    fc.bias_mu if fc.bias else None
                )

            else:
                logits = fc(x)

        return logits

    def random_dropconnect(self, mu, log_sigma):
        mask = torch.bernoulli(torch.full_like(mu, 1 - self.drop_prob))
        mu = mu * mask
        log_sigma = log_sigma * mask + (1 - mask) * (-10.0)
        return mu, log_sigma

    def signentropy_dropconnect_train(self, mu, log_sigma, current_epoch, total_epochs):
        sigma = torch.exp(log_sigma)
        entropy = calculate_entropy_dist(mu, sigma)
        E = entropy.flatten()

        E_mean = torch.quantile(E, 1 - self.drop_prob)

        score = self.alpha * (E - E_mean)
        p_drop = torch.sigmoid(score)
        p_drop = torch.clamp(p_drop, 0.0, 1.0)

        cos_term = 0.5 * (
            1 + torch.cos(
                torch.tensor(
                    np.pi * current_epoch / total_epochs,
                    device=mu.device
                )
            )
        )

        annealing_factor = 1 - (cos_term ** 5)

        p_drop_annealed = annealing_factor * p_drop
        p_drop_annealed = p_drop_annealed.view_as(mu)

        masked_mu = mu * (1 - p_drop_annealed)
        masked_sigma = sigma * (1 - p_drop_annealed)
        masked_log_sigma = torch.log(masked_sigma + 1e-8)

        return masked_mu, masked_log_sigma



class BayesianResNet18_FC(nn.Module):
    def __init__(self, drop_type=None, drop_prob=0.2, alpha=50):
        super().__init__()
        self.backbone = models.resnet18(weights=None)

        # Optional: Re-declare conv1 for flexibility with input shape (CIFAR-10)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False) # channels 3 for cifar10 and 1 for mnist
        self.backbone.maxpool = nn.Identity()  # Optional for CIFAR-10 to preserve spatial resolution

        # Replace FC with Bayesian linear layer
        in_features = self.backbone.fc.in_features
        self.backbone.fc = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1, in_features=in_features, out_features=10)

        # Dropout, DropConnect, SignEntropy setup
        self.drop_type = drop_type  # Options: None, 'dropout', 'dropconnect', 'signentropy'
        self.drop_prob = drop_prob
        self.alpha = alpha  # May be used for sign entropy methods

        if self.drop_type == 'dropout':
            self.dropout_layer = nn.Dropout(p=self.drop_prob)

    def forward_features(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)

        x = self.backbone.avgpool(x)
        x = torch.flatten(x, 1)
        return x

    def forward(self, x, current_epoch=None, total_epochs=None):
        x = self.forward_features(x)
        fc = self.backbone.fc

        if self.training:
            if self.drop_type == 'dropconnect':
                masked_mu, _ = self.random_dropconnect(fc.weight_mu, fc.weight_log_sigma)
                logits = F.linear(x, masked_mu, fc.bias_mu if fc.bias else None)

            elif self.drop_type == 'signentropy':
                fc.weight_mu.data, fc.weight_log_sigma.data = self.signentropy_dropconnect_train(
                    fc.weight_mu.data, fc.weight_log_sigma.data, current_epoch, total_epochs
                )
                logits = fc(x)

            else:
                logits = fc(x)
                if self.drop_type == 'dropout':
                    logits = self.dropout_layer(logits)

        else:
            if self.drop_type == 'dropconnect':
                masked_mu = fc.weight_mu * (1 - self.drop_prob)
                logits = F.linear(x, masked_mu, fc.bias_mu if fc.bias else None)

            elif self.drop_type == 'signentropy':
                #scaled_mu, _ = self.signentropy_dropconnect_inference(fc.weight_mu, fc.weight_log_sigma)
                logits = F.linear(x, fc.weight_mu, fc.bias_mu if fc.bias else None) # no scaling on inference

            else:
                logits = fc(x)

        return logits



    # ---------- HELPER FUNCTIONS BELOW ----------

    def random_dropconnect(self, mu, log_sigma):
        mask = torch.bernoulli(torch.full_like(mu, 1 - self.drop_prob))
        mu = mu * mask
        log_sigma = log_sigma * mask + (1 - mask) * (-10.0)
        return mu, log_sigma


    def signentropy_dropconnect_train(self, mu, log_sigma, current_epoch, total_epochs):
        # Calculate entropy from mu and sigma
        sigma = torch.exp(log_sigma)
        entropy = calculate_entropy_dist(mu, sigma)
        E = entropy.flatten()

        # Compute E_mean using quantile (based on target drop ratio)
        E_mean = torch.quantile(E, 1 - self.drop_prob)

        # Compute score and sharpen with alpha
        score = self.alpha * (E - E_mean)
        p_drop = torch.sigmoid(score)

        #scaling p_drop to match drop ratio
        #mean_p = p_drop.mean()
        #p_drop = (p_drop / (mean_p + 1e-8)) * self.drop_prob # multiply by * bias

        p_drop = torch.clamp(p_drop, 0.0, 1.0)

        # Cosine annealing to gradually increase regularization
        #annealing_factor = 1 - 0.5 * (1 + torch.cos(torch.tensor(np.pi * current_epoch / total_epochs, device=mu.device)))
        cos_term =  0.5 * (1 + torch.cos(torch.tensor(np.pi * current_epoch / total_epochs, device=mu.device)))
        gamma = 0.1 ## power term
        annealing_factor = 1 - (cos_term ** gamma)
        p_drop_annealed = annealing_factor * p_drop

        # Reshape p_drop_annealed back to original mu shape
        p_drop_annealed = p_drop_annealed.view_as(mu)

        # Attenuate mean and standard deviation according to p_drop_annealed
        masked_mu = mu * (1 - p_drop_annealed)
        masked_sigma = sigma * (1 - p_drop_annealed)
        masked_log_sigma = torch.log(masked_sigma + 1e-8)

        return masked_mu, masked_log_sigma

    def signentropy_dropconnect_inference(self, mu, log_sigma):
        E = torch.abs(mu)
        E_mean = torch.quantile(E.flatten(), 1 - self.drop_prob)

        score = self.alpha * (E - E_mean)
        p_drop = torch.sigmoid(score)
        p_drop = torch.clamp(p_drop, 0.0, 1.0)

        p_mean = p_drop.mean()

        # Variance-preserving scaling
        scale_factor = (1 - p_drop) * (1 + p_drop - p_mean) / (1 - p_mean + 1e-8)

        scaled_mu = mu * scale_factor
        sigma = torch.exp(log_sigma)
        scaled_sigma = sigma * scale_factor
        scaled_log_sigma = torch.log(scaled_sigma + 1e-8)

        return scaled_mu, scaled_log_sigma


# -------------- Training ----------------
def train_model(model, train_loader, optimizer, criterion, device, total_epochs):
    model.train()
    for epoch in range(1, total_epochs + 1):
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()

            output = model(X, current_epoch=epoch, total_epochs=total_epochs)  # Pass epochs info
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()


# -------------- Compute Entropy from Predictions ----------------
def compute_entropy_from_preds(preds_entropy):
    preds_entropy = np.array(preds_entropy)

    # If preds_entropy is 1D (e.g., (n,)), reshape it to (n, 1)
    if preds_entropy.ndim == 1:
        preds_entropy = preds_entropy[:, np.newaxis]

    # If preds_entropy is 2D (num_samples, num_points) and binary 0/1
    if preds_entropy.ndim == 2:
        num_samples, num_points = preds_entropy.shape
        num_classes = int(preds_entropy.max() + 1)
        preds_entropy_onehot = np.eye(num_classes)[preds_entropy.astype(int)]  # (num_samples, num_points, num_classes)
    else:
        preds_entropy_onehot = preds_entropy

    # Now preds_entropy_onehot is (num_samples, num_points, num_classes)
    num_samples, num_points, num_classes = preds_entropy_onehot.shape

    # Average over MC samples
    avg_preds = preds_entropy_onehot.mean(axis=0)  # shape (num_points, num_classes)

    # Calculate entropy
    with np.errstate(divide='ignore', invalid='ignore'):
        entropy = -np.sum(avg_preds * np.log2(avg_preds + 1e-10), axis=1)  # (num_points,)

    # Normalize entropy between 0 and 1
    entropy /= np.log2(num_classes)
    entropy = np.clip(entropy, 0, 1)

    return entropy

# -------------- Compute ECE ----------------
def compute_ece(y_true, y_prob, n_bins=15):
    """
    Multi-class version of ECE.
    Computes ECE for each class independently (one-vs-rest) and averages.
    """
    n_classes = y_prob.shape[1]
    ece_list = []

    for class_idx in range(n_classes):
        y_true_binary = (y_true == class_idx).astype(int)
        y_prob_class = y_prob[:, class_idx]

        prob_true, prob_pred = calibration_curve(y_true_binary, y_prob_class, n_bins=n_bins, strategy='uniform')

        if len(prob_true) > 0 and len(prob_pred) > 0:  # avoid empty bins
            ece = np.abs(prob_pred - prob_true).mean()
            ece_list.append(ece)

    return np.mean(ece_list)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --------------------
# Helper Functions
# --------------------

def bayesian_predict_with_uncertainty_mask(model, X_loader, num_samples=100, dropconnect_type=None, dropconnect_prob=0.1):
    model.eval()
    all_samples = []

    with torch.no_grad():
        for _ in range(num_samples):
            model_copy = copy.deepcopy(model)
            model_state = model_copy.state_dict()

            for name in model_state.keys():
                if "weight_mu" in name:
                    weight_mu = model_state[name]
                    weight_log_sigma_name = name.replace("weight_mu", "weight_log_sigma")
                    weight_log_sigma = model_state.get(weight_log_sigma_name, torch.zeros_like(weight_mu))
                    weight_sigma = torch.exp(weight_log_sigma)

                    sampled_weight = weight_mu + torch.randn_like(weight_mu) * weight_sigma

                    if dropconnect_type == "random":
                        drop_mask = (torch.rand_like(sampled_weight) > dropconnect_prob).float()
                        sampled_weight = sampled_weight * drop_mask
                    elif dropconnect_type == "entropy":
                        mu_np = weight_mu.detach().cpu().numpy()
                        sigma_np = weight_sigma.detach().cpu().numpy()
                        entropy = calculate_entropy_dist(mu_np, sigma_np)
                        flat_entropy = entropy.flatten()
                        k = int(len(flat_entropy) * dropconnect_prob)
                        topk_indices = np.argsort(flat_entropy)[-k:]
                        mask_np = np.ones_like(flat_entropy, dtype=bool)
                        mask_np[topk_indices] = False
                        mask_tensor = torch.tensor(mask_np.reshape(weight_mu.shape), device=weight_mu.device)
                        sampled_weight = sampled_weight * mask_tensor

                    model_state[name] = sampled_weight

            model_copy.load_state_dict(model_state, strict=False)
            model_copy.to(device)

            preds = []
            for images, _ in X_loader:
                images = images.to(device)
                outputs = outputs = torch.softmax(model(images), dim=1) #torch.sigmoid(model_copy(images))
                preds.append(outputs.cpu().numpy())

            preds = np.concatenate(preds, axis=0)
            all_samples.append(preds)

    return np.stack(all_samples, axis=0)

def bayesian_predict_with_uncertainty_expectationScaling(model, X_loader, num_samples=100, dropconnect_type=None, dropconnect_prob=0.1):
    """
    Perform MC sampling with expectation scaling (no masking at test time).
    """
    model.eval()
    all_samples = []

    with torch.no_grad():
        for _ in range(num_samples):
            model_copy = copy.deepcopy(model)
            model_state = model_copy.state_dict()

            for name in model_state.keys():
                if "weight_mu" in name:
                    weight_mu = model_state[name]
                    weight_log_sigma_name = name.replace("weight_mu", "weight_log_sigma")
                    weight_log_sigma = model_state.get(weight_log_sigma_name, torch.zeros_like(weight_mu))
                    weight_sigma = torch.exp(weight_log_sigma)

                    # 1. Sample from posterior
                    sampled_weight = weight_mu + torch.randn_like(weight_mu) * weight_sigma

                    # 2. Apply Expectation Scaling
                    if dropconnect_type in ["random", "entropy"]:
                        sampled_weight = sampled_weight * (1 - dropconnect_prob)

                    # 3. Update weight
                    model_state[name] = sampled_weight

            model_copy.load_state_dict(model_state, strict=False)
            model_copy.to(device)

            preds = []
            for images, _ in X_loader:
                images = images.to(device)
                outputs = torch.softmax(model_copy(images), dim=1)  # Note: model_copy here!
                preds.append(outputs.cpu().numpy())

            preds = np.concatenate(preds, axis=0)
            all_samples.append(preds)

    return np.stack(all_samples, axis=0)


### no expectation scaling for uncertainity evaluation
def bayesian_predict_with_uncertainty_mask1(model, X_loader, num_samples=100, dropconnect_type=None, dropconnect_prob=0.1):
    """
    Perform MC sampling:
    - Dropout is handled automatically via model.train().
    - For DropConnect types, manually mask sampled weights.
    """
    all_samples = []

    for _ in range(num_samples):
        model_copy = copy.deepcopy(model)
        #model_copy.train()  # Enable dropout stochasticity
        model_copy.eval()
        model_state = model_copy.state_dict()

        # MANUAL masking if dropconnect
        if dropconnect_type in ["random", "entropy"]:
            for name in model_state.keys():
                if "weight_mu" in name:
                    weight_mu = model_state[name]
                    weight_log_sigma_name = name.replace("weight_mu", "weight_log_sigma")
                    weight_log_sigma = model_state.get(weight_log_sigma_name, torch.zeros_like(weight_mu))
                    weight_sigma = torch.exp(weight_log_sigma)

                    sampled_weight = weight_mu + torch.randn_like(weight_mu) * weight_sigma

                    # if dropconnect_type == "random":
                    #     drop_mask = (torch.rand_like(sampled_weight) > dropconnect_prob).float()
                    #     sampled_weight = sampled_weight * drop_mask

                    # elif dropconnect_type == "entropy":
                    #     mu_np = weight_mu.detach().cpu().numpy()
                    #     sigma_np = weight_sigma.detach().cpu().numpy()
                    #     entropy = calculate_entropy_dist(mu_np, sigma_np)
                    #     flat_entropy = entropy.flatten()
                    #     k = int(len(flat_entropy) * dropconnect_prob)
                    #     topk_indices = np.argsort(flat_entropy)[-k:]
                    #     mask_np = np.ones_like(flat_entropy, dtype=bool)
                    #     mask_np[topk_indices] = False
                    #     mask_tensor = torch.tensor(mask_np.reshape(weight_mu.shape), device=weight_mu.device)
                    #     sampled_weight = sampled_weight * mask_tensor

                    model_state[name] = sampled_weight

            model_copy.load_state_dict(model_state, strict=False)

        model_copy.to(device)

        preds = []
        with torch.no_grad():
            for images, _ in X_loader:
                images = images.to(device)
                outputs = torch.softmax(model_copy(images), dim=1)
                preds.append(outputs.cpu().numpy())

        preds = np.concatenate(preds, axis=0)
        all_samples.append(preds)

    return np.stack(all_samples, axis=0)

def bayesian_predict_with_uncertainty_nologits(model, X_loader, num_samples=100, dropconnect_type=None, dropconnect_prob=0.1):
    """
    Perform MC sampling:
    - Dropout is handled automatically via model.train().
    - For DropConnect types, manually mask sampled weights.
    """
    all_samples = []

    for _ in range(num_samples):
        model_copy = copy.deepcopy(model)
        #model_copy.train()  # Enable dropout stochasticity
        model_copy.eval()
        model_state = model_copy.state_dict()

        # MANUAL masking if dropconnect
        for name in model_state.keys():
            if "weight_mu" in name:
                weight_mu = model_state[name]
                weight_log_sigma_name = name.replace("weight_mu", "weight_log_sigma")
                weight_log_sigma = model_state.get(weight_log_sigma_name, torch.zeros_like(weight_mu))
                weight_sigma = torch.exp(weight_log_sigma)

                sampled_weight = weight_mu + torch.randn_like(weight_mu) * weight_sigma

                model_state[name] = sampled_weight

        model_copy.load_state_dict(model_state, strict=False)

        model_copy.to(device)

        preds = []
        with torch.no_grad():
            for images, _ in X_loader:
                images = images.to(device)
                outputs = torch.softmax(model_copy(images), dim=1)
                preds.append(outputs.cpu().numpy())

        preds = np.concatenate(preds, axis=0)
        all_samples.append(preds)

    return np.stack(all_samples, axis=0)


def bayesian_predict_with_uncertainty_latest(model, X_loader, num_samples=100, dropconnect_type=None, dropconnect_prob=0.1, return_logits=False):
    """
    Perform MC sampling:
    - Dropout is handled automatically via model.train().
    - For DropConnect types, manually mask sampled weights.
    - Optionally return logits or probabilities.
    """
    all_samples = []

    for _ in range(num_samples):
        model_copy = copy.deepcopy(model)
        model_copy.eval()
        model_state = model_copy.state_dict()

        # Manual DropConnect sampling if required
        for name in model_state.keys():
            if "weight_mu" in name:
                weight_mu = model_state[name]
                weight_log_sigma_name = name.replace("weight_mu", "weight_log_sigma")
                weight_log_sigma = model_state.get(weight_log_sigma_name, torch.zeros_like(weight_mu))
                weight_sigma = torch.exp(weight_log_sigma)

                sampled_weight = weight_mu + torch.randn_like(weight_mu) * weight_sigma
                model_state[name] = sampled_weight

        model_copy.load_state_dict(model_state, strict=False)
        model_copy.to(device)

        preds = []
        with torch.no_grad():
            for images, _ in X_loader:
                images = images.to(device)
                outputs = model_copy(images)

                if not return_logits:
                    outputs = torch.softmax(outputs, dim=1)

                preds.append(outputs.cpu().numpy())

        preds = np.concatenate(preds, axis=0)
        all_samples.append(preds)

    return np.stack(all_samples, axis=0)  # Shape: (num_samples, N, num_classes)

In [ ]:
#### Temperature Scaling

import torch.nn as nn
import torch.nn.functional as F

class ModelWithTemperature(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)  # Initial temp > 1

    def forward(self, input):
        logits = self.model(input)
        return self.temperature_scale(logits)

    def temperature_scale(self, logits):
        # Apply temperature scaling
        return logits / self.temperature

    def set_temperature(self, valid_loader, device):
        self.to(device)
        self.eval()
        logits_list = []
        labels_list = []

        with torch.no_grad():
            for input, label in valid_loader:
                input = input.to(device)
                logits = self.model(input)
                logits_list.append(logits)
                labels_list.append(label.to(device))

        logits = torch.cat(logits_list)
        labels = torch.cat(labels_list)

        # Optimize temperature using validation NLL
        nll_criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def eval():
            optimizer.zero_grad()
            loss = nll_criterion(self.temperature_scale(logits), labels)
            loss.backward()
            return loss

        optimizer.step(eval)
        print(f"Optimal temperature: {self.temperature.item():.4f}")
        return self


def brier_score(y_true, y_prob):
    return np.mean((y_prob - y_true) ** 2)


transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),    # Randomly flip images horizontally
    transforms.RandomCrop(32, padding=4),  # Random crop with padding
    transforms.RandomRotation(15),         # Small random rotations
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Color variation
    transforms.ToTensor(),
])


transform_test = transforms.Compose([
    transforms.ToTensor(),
])

In [ ]:
# CutMix or MixUp (Advanced Data Augmentation)

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(x.size()[0]).to(x.device)
    y_a, y_b = y, y[rand_index]
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[rand_index, :, bbx1:bbx2, bby1:bby2]
    return x, y_a, y_b, lam

def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

In [ ]:
# --------------------
# Main Experiment all images old code
# --------------------


batch_size = 256
target_img_size = (32,32)

# CIFAR-10 dataset loading
"""
num_epochs = 80


transform_train = transforms.Compose([
    transforms.Resize(target_img_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])  # Channel-wise normalization
    #transforms.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0)
])

transform_test = transforms.Compose([
    transforms.Resize(target_img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])
])


dataset_name = 'cifar10'
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
# ### cifar10
"""


### MNIST
"""
transform_train = transforms.Compose([
    transforms.Resize((32, 32)),  # Optional: resize to match expected input size
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Fashion-MNIST is grayscale (1 channel)
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

dataset_name = 'mnist'
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform_test)
"""
### MNIST

####Imagenette
#dataset_name = 'imagenette'
#from fastdownload import FastDownload
#fd = FastDownload()
#imagenette_path = fd.get("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz")
#data_dir = imagenette_path
#trainset = torchvision.datasets.ImageFolder(root=f"{data_dir}/train", transform=transform_train)
#testset = torchvision.datasets.ImageFolder(root=f"{data_dir}/val", transform=transform_test)
####Imagenette


# Fashion-MNIST dataset loading

dataset_name = 'fashionmnist'
num_epochs = 40
transform_train = transforms.Compose([
    transforms.Resize((32, 32)),  # Optional: resize to match expected input size
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Fashion-MNIST is grayscale (1 channel)
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

trainset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform_test)
# Fashion-MNIST dataset loading


# Subsample random indices without replacement
subset_indices = torch.randperm(len(trainset))#[:11000]
# Create a Subset of the trainset
train_subset = torch.utils.data.Subset(trainset, subset_indices)
val_ratio = 0.1
total_subset = len(subset_indices)
val_size = int(val_ratio * total_subset)
train_size = total_subset - val_size

# Create two subsets
train_subset_split, val_subset_split = random_split(train_subset, [train_size, val_size])
# Create the trainloader from the subset
#trainloader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
trainloader = torch.utils.data.DataLoader(train_subset_split, batch_size=batch_size, shuffle=True, num_workers=0)
valloader = torch.utils.data.DataLoader(val_subset_split, batch_size=batch_size, shuffle=False, num_workers=0)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
# concrete dropout for VGG16
class ConcreteVGG16_FC(nn.Module):
    def __init__(
        self,
        drop_prob=0.3,
        alpha=50,                 # unused, kept for compatibility
        in_channels=1,            # 1 for MNIST/FashionMNIST, 3 for CIFAR10
        num_classes=10,
        weight_regularizer=1e-6,
        dropout_regularizer=1e-5,
        concrete_temperature=0.1
    ):
        super().__init__()

        self.backbone = models.vgg16(weights=None)

        # Change first conv layer for grayscale or RGB input
        self.backbone.features[0] = nn.Conv2d(
            in_channels,
            64,
            kernel_size=3,
            stride=1,
            padding=1
        )

        # For 32x32 images, use compact pooling
        self.backbone.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Replace huge VGG classifier with simple final linear classifier
        self.feature_dim = 512
        self.classifier = nn.Linear(self.feature_dim, num_classes)

        self.drop_type = "concrete"
        self.drop_prob = drop_prob
        self.alpha = alpha

        # Concrete Dropout learnable probability
        init_p = np.clip(drop_prob, 1e-6, 1.0 - 1e-6)
        init_logit = np.log(init_p) - np.log(1.0 - init_p)

        self.concrete_p_logit = nn.Parameter(
            torch.tensor([init_logit], dtype=torch.float32)
        )

        self.concrete_temperature = concrete_temperature
        self.weight_regularizer = weight_regularizer
        self.dropout_regularizer = dropout_regularizer

        self.cd_regularization = torch.tensor(0.0)

    def forward_features(self, x):
        x = self.backbone.features(x)
        x = self.backbone.avgpool(x)
        x = torch.flatten(x, 1)
        return x

    def forward(self, x, current_epoch=None, total_epochs=None):
        """
        current_epoch and total_epochs are ignored.
        They are included only so the existing training loop does not break.
        """

        x = self.forward_features(x)

        input_dimensionality = x[0].numel()

        # Concrete Dropout on the final feature vector before classifier
        # Kept stochastic in both train and eval for MC Concrete Dropout
        x = self.concrete_dropout(x)

        logits = self.classifier(x)

        self.cd_regularization = self.concrete_dropout_regularization(
            self.classifier,
            input_dimensionality
        )

        return logits

    def concrete_dropout(self, x):
        eps = 1e-7

        p = torch.sigmoid(self.concrete_p_logit)
        p = torch.clamp(p, eps, 1.0 - eps)

        unif_noise = torch.rand_like(x)

        drop_prob = (
            torch.log(p + eps)
            - torch.log(1.0 - p + eps)
            + torch.log(unif_noise + eps)
            - torch.log(1.0 - unif_noise + eps)
        )

        drop_prob = torch.sigmoid(drop_prob / self.concrete_temperature)

        random_tensor = 1.0 - drop_prob
        retain_prob = 1.0 - p

        x = x * random_tensor
        x = x / retain_prob

        return x

    def concrete_dropout_regularization(self, layer, input_dimensionality):
        eps = 1e-7

        p = torch.sigmoid(self.concrete_p_logit)
        p = torch.clamp(p, eps, 1.0 - eps)

        sum_of_square = 0.0
        for param in layer.parameters():
            sum_of_square = sum_of_square + torch.sum(param ** 2)

        weights_regularizer = (
            self.weight_regularizer * sum_of_square / (1.0 - p)
        )

        dropout_regularizer = p * torch.log(p)
        dropout_regularizer = dropout_regularizer + (
            1.0 - p
        ) * torch.log(1.0 - p)

        dropout_regularizer = (
            dropout_regularizer
            * self.dropout_regularizer
            * input_dimensionality
        )

        regularization = weights_regularizer + dropout_regularizer

        return regularization.squeeze()

    def get_concrete_dropout_prob(self):
        return torch.sigmoid(
            self.concrete_p_logit
        ).detach().cpu().item()

In [ ]:
#### Concrete Dropout for Resnet18

class ConcreteResNet18_FC(nn.Module):
    def __init__(
        self,
        drop_prob=0.2,
        alpha=50,                 # unused, kept only for compatibility
        in_channels=3,            # 1 for MNIST/FashionMNIST, 3 for CIFAR10
        num_classes=10,
        weight_regularizer=1e-6,
        dropout_regularizer=1e-5,
        concrete_temperature=0.1
    ):
        super().__init__()

        self.backbone = models.resnet18(weights=None)

        # Same as BayesianResNet18_FC
        self.backbone.conv1 = nn.Conv2d(
            in_channels,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.backbone.maxpool = nn.Identity()

        # Important: normal deterministic final layer
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

        self.drop_type = "concrete"
        self.drop_prob = drop_prob
        self.alpha = alpha

        # Concrete Dropout parameters
        init_p = np.clip(drop_prob, 1e-6, 1.0 - 1e-6)
        init_logit = np.log(init_p) - np.log(1.0 - init_p)

        self.concrete_p_logit = nn.Parameter(
            torch.tensor([init_logit], dtype=torch.float32)
        )

        self.concrete_temperature = concrete_temperature
        self.weight_regularizer = weight_regularizer
        self.dropout_regularizer = dropout_regularizer

        self.cd_regularization = torch.tensor(0.0)

    def forward_features(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)

        x = self.backbone.avgpool(x)
        x = torch.flatten(x, 1)

        return x

    def forward(self, x, current_epoch=None, total_epochs=None):
        """
        current_epoch and total_epochs are ignored.
        They are included only so your existing training loop does not break.
        """

        x = self.forward_features(x)

        input_dimensionality = x[0].numel()

        # Concrete Dropout remains stochastic in both train and eval mode.
        # This is needed for MC Concrete Dropout evaluation.
        x = self.concrete_dropout(x)

        logits = self.backbone.fc(x)

        self.cd_regularization = self.concrete_dropout_regularization(
            self.backbone.fc,
            input_dimensionality
        )

        return logits

    def concrete_dropout(self, x):
        eps = 1e-7

        p = torch.sigmoid(self.concrete_p_logit)
        p = torch.clamp(p, eps, 1.0 - eps)

        unif_noise = torch.rand_like(x)

        drop_prob = (
            torch.log(p + eps)
            - torch.log(1.0 - p + eps)
            + torch.log(unif_noise + eps)
            - torch.log(1.0 - unif_noise + eps)
        )

        drop_prob = torch.sigmoid(drop_prob / self.concrete_temperature)

        random_tensor = 1.0 - drop_prob
        retain_prob = 1.0 - p

        x = x * random_tensor
        x = x / retain_prob

        return x

    def concrete_dropout_regularization(self, layer, input_dimensionality):
        eps = 1e-7

        p = torch.sigmoid(self.concrete_p_logit)
        p = torch.clamp(p, eps, 1.0 - eps)

        # This follows the original GitHub-style implementation:
        # sum over parameters of the deterministic nn.Linear layer.
        sum_of_square = 0.0
        for param in layer.parameters():
            sum_of_square = sum_of_square + torch.sum(param ** 2)

        weights_regularizer = (
            self.weight_regularizer * sum_of_square / (1.0 - p)
        )

        dropout_regularizer = p * torch.log(p)
        dropout_regularizer = dropout_regularizer + (
            1.0 - p
        ) * torch.log(1.0 - p)

        dropout_regularizer = (
            dropout_regularizer
            * self.dropout_regularizer
            * input_dimensionality
        )

        regularization = weights_regularizer + dropout_regularizer

        return regularization.squeeze()

    def get_concrete_dropout_prob(self):
        return torch.sigmoid(
            self.concrete_p_logit
        ).detach().cpu().item()

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR
from torch.nn.utils import clip_grad_value_

initial_dropout_ratio = 0.3

concrete_model = ConcreteResNet18_FC(
    drop_prob=initial_dropout_ratio
    ,
    in_channels=3,      # use 3 for CIFAR10
    num_classes=10,
    weight_regularizer=1e-6,
    dropout_regularizer=1e-5,
    concrete_temperature=0.1
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    concrete_model.parameters(),
    lr=1e-3,
    weight_decay=0.0
)

scheduler = OneCycleLR(
    optimizer,
    max_lr=3e-3,
    epochs=num_epochs,
    steps_per_epoch=len(trainloader),
    pct_start=0.3,
    anneal_strategy='cos'
)

best_val_loss = float("inf")
best_model_state = None
epochs_no_improve = 0
patience = 10

concrete_train_losses = []
concrete_val_losses = []
concrete_val_f1s = []
concrete_ps = []

for epoch in range(1, num_epochs + 1):
    concrete_model.train()
    epoch_loss = 0.0
    batch_count = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        # CutMix, same as your BNN training code
        images, targets1, targets2, lam = cutmix_data(images, labels, alpha=1.0)

        optimizer.zero_grad()

        outputs = concrete_model(
            images,
            current_epoch=epoch,
            total_epochs=num_epochs
        )

        ce_loss = (
            lam * criterion(outputs, targets1)
            + (1.0 - lam) * criterion(outputs, targets2)
        )

        # Important: add Concrete Dropout regularization
        loss = ce_loss + concrete_model.cd_regularization

        loss.backward()

        clip_grad_value_(concrete_model.parameters(), clip_value=0.1)

        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        batch_count += 1

    avg_train_loss = epoch_loss / batch_count

    # ---------------- Validation ----------------
    concrete_model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in valloader:
            images, labels = images.to(device), labels.to(device)

            outputs = concrete_model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(valloader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    current_p = concrete_model.get_concrete_dropout_prob()

    concrete_train_losses.append(avg_train_loss)
    concrete_val_losses.append(avg_val_loss)
    concrete_val_f1s.append(macro_f1)
    concrete_ps.append(current_p)

    print(
        f"[concrete] Epoch {epoch}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val F1: {macro_f1:.4f} | "
        f"Concrete p: {current_p:.6f} | "
        f"LR: {scheduler.get_last_lr()[0]:.2e}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = copy.deepcopy(concrete_model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            # break

if best_model_state is not None:
    concrete_model.load_state_dict(best_model_state)

concrete_model.eval()

learned_concrete_p = concrete_model.get_concrete_dropout_prob()

print("\nConcrete Dropout training completed.")
print(f"Best Val Loss: {best_val_loss:.4f}")
print(f"Learned Concrete Dropout p = {learned_concrete_p:.6f}")

In [ ]:
import copy

# Make sure concrete_model is already trained
concrete_model.eval()

models_dict = {}

# Add trained concrete dropout model to models_dict
models_dict["concrete"] = copy.deepcopy(concrete_model).to(device)
models_dict["concrete"].eval()

# Optional: reorder models_dict
desired_order = ["concrete", "plain", "dropout", "dropconnect", "entropy"]

models_dict = {
    name: models_dict[name]
    for name in desired_order
    if name in models_dict
}

print("Final models_dict keys:", list(models_dict.keys()))

print(
    "Concrete learned p:",
    models_dict["concrete"].get_concrete_dropout_prob()
)

In [ ]:
# For L2
models_dict = {
    "plainL2_0.01": BayesianResNet18_FC(drop_type=None, drop_prob=dropout_ratio).to(device),
    "plainL2_0.1": BayesianResNet18_FC(drop_type=None, drop_prob=dropout_ratio).to(device),
    "plainL2_0.5": BayesianResNet18_FC(drop_type=None, drop_prob=dropout_ratio).to(device),
    "plainL2_1": BayesianResNet18_FC(drop_type=None, drop_prob=dropout_ratio).to(device)
}

In [ ]:
num_epochs = 60
dropout_ratio = 0.5
print(dataset_name + " " + str(num_epochs))
#print("concrete dropout ratio: " + str(learned_concrete_p))
print("dropout ratio: " + str(dropout_ratio))

In [ ]:
#### Training
from torch.optim.lr_scheduler import OneCycleLR
from torch.nn.utils import clip_grad_value_


#### Resnet18
"""
models_dict = {
    #"plain": BayesianResNet18_FC(drop_type=None, drop_prob=dropout_ratio).to(device),
    #"dropout": BayesianResNet18_FC(drop_type="dropout", drop_prob=dropout_ratio).to(device),
    #"dropconnect": BayesianResNet18_FC(drop_type="dropconnect", drop_prob=dropout_ratio).to(device),
    #"entropy": BayesianResNet18_FC(drop_type="signentropy", drop_prob=dropout_ratio).to(device)
}
"""

#### VGG16
models_dict = {
    "plain": BayesianVGG16_FC(
        drop_type=None,
        drop_prob=dropout_ratio,
        num_classes=10,
        in_channels=1
    ).to(device),

    "dropout": BayesianVGG16_FC(
        drop_type="dropout",
        drop_prob=dropout_ratio,
        num_classes=10,
        in_channels=1
    ).to(device),

    "dropconnect": BayesianVGG16_FC(
        drop_type="dropconnect",
        drop_prob=dropout_ratio,
        num_classes=10,
        in_channels=1
    ).to(device),

    "entropy": BayesianVGG16_FC(
        drop_type="signentropy",
        drop_prob=dropout_ratio,
        num_classes=10,
        in_channels=1
    ).to(device),

    "concrete": ConcreteVGG16_FC(
        drop_prob=dropout_ratio,
        num_classes=10,
        in_channels=1
    ).to(device),
}

# Set early stopping parameters
patience = 10
val_losses = {name: [] for name in models_dict}
val_f1s = {name: [] for name in models_dict}
train_losses = {name: [] for name in models_dict}

for name, model in models_dict.items():

    #l2_lambda = float(name.split('_')[-1])
    #print(f"Model: {name} | lambda: {l2_lambda}")
    # Initialize optimizer with weight decay (AdamW for proper weight decay handling)
    optimizer = optim.AdamW(model.parameters(),
                          lr=1e-3,  # Base LR will be overridden by scheduler
                          weight_decay=0.0)  # Weight decay for L2 regularization only

    # Initialize OneCycleLR scheduler
    scheduler = OneCycleLR(optimizer,
                         max_lr=3e-3,  # Peak learning rate
                         epochs=num_epochs,
                         steps_per_epoch=len(trainloader),
                         pct_start=0.3,  # 30% of cycle for warmup
                         anneal_strategy='cos')

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    model.to(device)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    lrs = []  # To track learning rate changes

    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_loss = 0.0
        batch_count = 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            # --- CutMix ---
            images, targets1, targets2, lam = cutmix_data(images, labels, alpha=1.0)

            optimizer.zero_grad()

            outputs = model(images, current_epoch=epoch, total_epochs=num_epochs)

            # Combined CutMix loss
            loss = lam * criterion(outputs, targets1) + (1 - lam) * criterion(outputs, targets2)
            loss.backward()

            clip_grad_value_(model.parameters(), clip_value=0.1)
            optimizer.step()
            scheduler.step()

            lrs.append(scheduler.get_last_lr()[0])
            epoch_loss += loss.item()
            batch_count += 1

        avg_loss = epoch_loss / batch_count

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in valloader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images) if "entropy" not in name.lower() else \
                        model(images, current_epoch=epoch, total_epochs=num_epochs)

                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_val_loss = val_loss / len(valloader)
        macro_f1 = f1_score(all_labels, all_preds, average='macro')

        # Store metrics
        val_losses[name].append(avg_val_loss)
        val_f1s[name].append(macro_f1)
        train_losses[name].append(avg_loss)

        print(f"[{name}] Epoch {epoch}/{num_epochs} | Train Loss: {avg_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | Val F1: {macro_f1:.4f} | "
              f"LR: {lrs[-1]:.2e}")

        # --- Early Stopping ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            best_model_state = copy.deepcopy(model.state_dict()) #model.state_dict()
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                #break  # (Optional: leave break commented if you want full epochs)

    # Load best model weights
    if best_model_state:
        model.load_state_dict(best_model_state)
    print(f"{name} completed. Best Val Loss: {best_val_loss:.4f}")

In [ ]:
#models_dict_dc_en = copy.deepcopy(models_dict)
#del models_dict

In [ ]:
# Check what is currently inside both dictionaries
"""
print("Current models_dict keys:", models_dict.keys())
print("Saved dropconnect/entropy keys:", models_dict_dc_en.keys())

# Add the trained dropconnect and entropy models back into models_dict
for model_name in ["dropconnect", "entropy"]:
    if model_name in models_dict_dc_en:
        models_dict[model_name] = models_dict_dc_en[model_name].to(device)
        models_dict[model_name].eval()
    else:
        print(f"{model_name} not found in models_dict_dc_en")

# Optional: reorder dictionary cleanly
ordered_names = ["plain", "dropout", "dropconnect", "entropy"]
models_dict = {
    name: models_dict[name]
    for name in ordered_names
    if name in models_dict
}

print("Final models_dict keys:", models_dict.keys())
"""

In [ ]:
import copy

# -------------------------------------------------------
# Add trained Concrete Dropout model into models_dict
# -------------------------------------------------------

# Make sure concrete_model is the trained ConcreteResNet18_FC model
concrete_model.eval()

models_dict["concrete"] = copy.deepcopy(concrete_model).to(device)
models_dict["concrete"].eval()

# Optional: reorder dictionary
desired_order = ["concrete", "plain", "dropout", "dropconnect", "entropy"]

models_dict = {
    name: models_dict[name]
    for name in desired_order
    if name in models_dict
}

print("Final models_dict keys:", list(models_dict.keys()))

if "concrete" in models_dict:
    print(
        "Concrete learned p:",
        models_dict["concrete"].get_concrete_dropout_prob()
    )

In [ ]:
#### Saving models

import os
import json
import torch
import numpy as np

# ------------------------------------------------------------------
# Set the output directory as needed
# ------------------------------------------------------------------
# Example output directory:
SAVE_DIR = r"results/fashion-mnist/0.5/"

# Alternative output directory example:
# SAVE_DIR = "results/fmnist_run_01"

os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(SAVE_DIR, "models_dict_checkpoint.pt")
METADATA_PATH = os.path.join(SAVE_DIR, "metadata.json")


def infer_drop_type_from_name(name):
    name = name.lower()

    if name == "plain" or name.startswith("plain"):
        return None
    elif name == "dropout":
        return "dropout"
    elif name == "dropconnect":
        return "dropconnect"
    elif name == "entropy" or name == "signentropy":
        return "signentropy"
    else:
        raise ValueError(f"Unknown model name: {name}")


def state_dict_to_cpu(model):
    """
    Saves only tensors on CPU.
    This avoids GPU/CPU loading problems later.
    """
    return {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }


def save_models_dict(
    models_dict,
    save_path,
    metadata_path=None,
    dropout_ratio=None,
    dataset_name=None,
    num_epochs=None,
    alpha_default=50
):
    checkpoint = {
        "state_dicts": {},
        "model_specs": {},
        "extra_info": {
            "dropout_ratio": float(dropout_ratio) if dropout_ratio is not None else None,
            "dataset_name": dataset_name,
            "num_epochs": int(num_epochs) if num_epochs is not None else None,
            "alpha_default": float(alpha_default),
        }
    }

    for name, model in models_dict.items():
        checkpoint["state_dicts"][name] = state_dict_to_cpu(model)

        checkpoint["model_specs"][name] = {
            "class_name": "BayesianResNet18_FC",
            "drop_type": infer_drop_type_from_name(name),
            "drop_prob": float(getattr(model, "drop_prob", dropout_ratio)),
            "alpha": float(getattr(model, "alpha", alpha_default)),
        }

    torch.save(checkpoint, save_path)

    if metadata_path is not None:
        metadata = {
            "saved_models": list(models_dict.keys()),
            "model_specs": checkpoint["model_specs"],
            "extra_info": checkpoint["extra_info"],
            "checkpoint_file": os.path.basename(save_path),
        }

        with open(metadata_path, "w") as f:
            json.dump(metadata, f, indent=4)

    print(f"Saved models_dict checkpoint to:\n{save_path}")

    if metadata_path is not None:
        print(f"Saved metadata to:\n{metadata_path}")


save_models_dict(
    models_dict=models_dict,
    save_path=CHECKPOINT_PATH,
    metadata_path=METADATA_PATH,
    dropout_ratio=dropout_ratio,
    dataset_name=dataset_name,
    num_epochs=num_epochs,
    alpha_default=50
)

In [ ]:
##### Testing the F1 is testset
for name, model in models_dict.items():
  all_preds = []
  all_labels = []

  model.eval()
  with torch.no_grad():
      for images, labels in testloader:
          images, labels = images.to(device), labels.to(device)
          outputs = model(images) if "entropy" not in name.lower() else \
                  model(images, current_epoch=epoch, total_epochs=num_epochs)

          loss = criterion(outputs, labels)
          val_loss += loss.item()
          preds = outputs.argmax(dim=1)
          all_preds.extend(preds.cpu().numpy())
          all_labels.extend(labels.cpu().numpy())

  macro_f1 = f1_score(all_labels, all_preds, average='macro')

  print(f"[{name}] Val F1: {macro_f1:.4f}")

In [ ]:
#### final save functions with concrete dropout model in models_dict

import os
import json
import torch

SAVE_DIR = r"results/fmnist_run_01"
os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(SAVE_DIR, "models_dict_checkpoint.pt")
METADATA_PATH = os.path.join(SAVE_DIR, "metadata.json")


def state_dict_to_cpu(model):
    return {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }


def get_model_spec(name, model):
    if name.lower() == "concrete" or getattr(model, "drop_type", None) == "concrete":
        return {
            "class_name": "ConcreteResNet18_FC",
            "drop_type": "concrete",
            "drop_prob": float(model.get_concrete_dropout_prob()),
            "alpha": float(getattr(model, "alpha", 50)),
        }

    else:
        return {
            "class_name": "BayesianResNet18_FC",
            "drop_type": getattr(model, "drop_type", None),
            "drop_prob": float(getattr(model, "drop_prob", dropout_ratio)),
            "alpha": float(getattr(model, "alpha", 50)),
        }


def save_models_dict(
    models_dict,
    save_path,
    metadata_path=None,
    dropout_ratio=None,
    dataset_name=None,
    num_epochs=None
):
    checkpoint = {
        "state_dicts": {},
        "model_specs": {},
        "extra_info": {
            "dropout_ratio": float(dropout_ratio) if dropout_ratio is not None else None,
            "dataset_name": dataset_name,
            "num_epochs": int(num_epochs) if num_epochs is not None else None,
        }
    }

    for name, model in models_dict.items():
        model.eval()

        checkpoint["state_dicts"][name] = state_dict_to_cpu(model)
        checkpoint["model_specs"][name] = get_model_spec(name, model)

    torch.save(checkpoint, save_path)

    if metadata_path is not None:
        metadata = {
            "saved_models": list(models_dict.keys()),
            "model_specs": checkpoint["model_specs"],
            "extra_info": checkpoint["extra_info"],
            "checkpoint_file": os.path.basename(save_path),
        }

        with open(metadata_path, "w") as f:
            json.dump(metadata, f, indent=4)

    print(f"Saved checkpoint to:\n{save_path}")

    if metadata_path is not None:
        print(f"Saved metadata to:\n{metadata_path}")

In [ ]:
def bayesian_predict_with_uncertainty(model, X_loader, num_samples=100, dropconnect_type=None, drop_prob=0.1, return_logits=False):
    """
    Perform MC sampling for Bayesian networks with DropConnect or SignEntropy regularization.

    Args:
        model: The trained model (BayesianResNet etc.)
        X_loader: DataLoader for input data
        num_samples: Number of MC samples
        dropconnect_type: "dropconnect", "signentropy", or None
        drop_prob: Drop probability used during training
        return_logits: If True, return raw logits; else return softmax probabilities

    Returns:
        A tuple of:
            - predictions: np.ndarray of shape (num_samples, N, num_classes)
            - labels: np.ndarray of shape (N,)
    """
    all_samples = []
    all_labels = None  # We'll collect this during the first MC sample

    for sample_idx in range(num_samples):
        model_copy = copy.deepcopy(model)
        model_copy.eval()
        model_state = model_copy.state_dict()

        for name in list(model_state.keys()):
            if "weight_mu" in name:
                weight_mu = model_state[name]
                weight_log_sigma_name = name.replace("weight_mu", "weight_log_sigma")
                weight_log_sigma = model_state.get(weight_log_sigma_name, torch.zeros_like(weight_mu))
                weight_sigma = torch.exp(weight_log_sigma)

                # Sample weights from posterior
                sampled_weight = weight_mu + torch.randn_like(weight_mu) * weight_sigma

                model_state[name] = sampled_weight

            elif "bias_mu" in name:
                bias_mu = model_state[name]
                bias_log_sigma_name = name.replace("bias_mu", "bias_log_sigma")
                bias_log_sigma = model_state.get(bias_log_sigma_name, torch.zeros_like(bias_mu))
                bias_sigma = torch.exp(bias_log_sigma)

                sampled_bias_mu = bias_mu + torch.randn_like(bias_mu) * bias_sigma
                model_state[name] = sampled_bias_mu
        model_copy.load_state_dict(model_state, strict=False)
        model_copy.to(device)

        preds = []
        labels_list = []
        with torch.no_grad():
            for images, labels in X_loader:
                images = images.to(device)
                outputs = model_copy(images)

                if not return_logits:
                    outputs = torch.softmax(outputs, dim=1)

                preds.append(outputs.cpu().numpy())
                if sample_idx == 0:
                    labels_list.append(labels.numpy())

        preds = np.concatenate(preds, axis=0)
        if sample_idx == 0:
            all_labels = np.concatenate(labels_list, axis=0)

        all_samples.append(preds)

    return np.stack(all_samples, axis=0), all_labels  # (num_samples, N, num_classes), (N,)

In [ ]:
##### without temp scaling calculate the metrics
num_mc_samples = 50

def compute_aece(y_true, y_prob, num_bins=15):
    """
    Computes Adaptive ECE for multiclass classification.

    Args:
        y_true: (N,) true labels
        y_prob: (N, C) predicted softmax probabilities
        num_bins: number of adaptive bins (default 15)

    Returns:
        AECE score
    """
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    # Get predicted class and confidence
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    correct = predictions == y_true
    n = len(y_true)

    # Sort by confidence
    sorted_indices = np.argsort(confidences)
    confidences = confidences[sorted_indices]
    correct = correct[sorted_indices]

    bin_size = n // num_bins
    aece = 0.0

    for i in range(num_bins):
        start = i * bin_size
        end = (i + 1) * bin_size if i < num_bins - 1 else n

        conf_bin = confidences[start:end]
        acc_bin = correct[start:end]

        if len(conf_bin) == 0:
            continue

        avg_conf = np.mean(conf_bin)
        avg_acc = np.mean(acc_bin)
        aece += (len(conf_bin) / n) * abs(avg_acc - avg_conf)

    return aece

metrics = {
    "plain": {"F1": [], "NLL": [], "Brier": [], "ECE": [], "AECE": []},
    "dropout": {"F1": [], "NLL": [], "Brier": [], "ECE": [], "AECE": []},
    "dropconnect": {"F1": [], "NLL": [], "Brier": [], "ECE": [], "AECE": []},
    "entropy": {"F1": [], "NLL": [], "Brier": [], "ECE": [], "AECE": []},
    "concrete": {"F1": [], "NLL": [], "Brier": [], "ECE": [], "AECE": []}
}

for name, model in models_dict.items():
    base_model = model.eval()  # No temperature scaling — just use model as-is

    # Get MC predictions (logits)
    preds_mc_logits, all_labels = bayesian_predict_with_uncertainty(
        base_model,
        testloader,
        num_samples=num_mc_samples,
        dropconnect_type=name,
        drop_prob=dropout_ratio,
        return_logits=True
    )

    preds_mc_binary = np.argmax(preds_mc_logits, axis=2)
    uncertainty = np.mean(compute_entropy_from_preds(preds_mc_binary))

    for mc_idx in range(num_mc_samples):
        logits = preds_mc_logits[mc_idx]  # (N, num_classes)

        # Apply softmax directly (no temperature scaling)
        preds = torch.softmax(torch.from_numpy(logits), dim=1).numpy()

        y_pred = np.argmax(preds, axis=1)

        f1 = f1_score(all_labels, y_pred, average='macro')
        nll = -np.mean(np.log(preds[np.arange(len(all_labels)), all_labels] + 1e-10))
        true_labels_onehot = np.eye(preds.shape[1])[all_labels]
        brier_score = np.mean(np.sum((preds - true_labels_onehot) ** 2, axis=1))
        ece = compute_ece(y_true=all_labels, y_prob=preds)
        aece = compute_aece(y_true=all_labels, y_prob=preds)

        print(f"Model: {name.upper()} | F1: {f1:.4f} | Uncertainty Label: {uncertainty:.4f} & NLL: {nll:.4f} & Brier:{brier_score:.4f} | ECE: {ece:.4f} AECE: {aece:.4f}")
        metrics[name]["F1"].append(f1)
        metrics[name]["NLL"].append(nll)
        metrics[name]["Brier"].append(brier_score)
        metrics[name]["ECE"].append(ece)
        metrics[name]["AECE"].append(aece)


    print(f"Model {name.upper()} finished MC sampling.")

In [ ]:
import numpy as np

for name in metrics:
    print(f"\nModel: {name}")
    for metric in ["F1", "NLL", "Brier", "ECE", "AECE"]:
        values = metrics[name][metric]
        if len(values) == 0:
            print(f"{metric}: No data available.")
        else:
            values = np.array(values)
            mean_val = np.mean(values)
            median_val = np.median(values)
            print(f"{metric}: Mean = {mean_val:.4f}, Median = {median_val:.4f}")

In [ ]:
#### Measuring Epistemic and Aleatoric Uncertainity


def compute_classification_uncertainty_from_mc(
    preds_mc,
    input_type="logits",
    normalize=True,
    eps=1e-12
):
    """
    Computes total, aleatoric, and epistemic uncertainty for classification
    from MC predictions.

    Args:
        preds_mc:
            Shape = (num_mc_samples, N, num_classes)

            If input_type="logits", preds_mc contains raw logits.
            If input_type="probs", preds_mc contains softmax probabilities.

        input_type:
            "logits" or "probs"

        normalize:
            If True, divide entropy values by log(num_classes), so values are in [0, 1].

        eps:
            Small value for numerical stability.

    Returns:
        uncertainty_dict with:
            - predictive_probs: mean probability, shape (N, C)
            - total_uncertainty_per_sample
            - aleatoric_uncertainty_per_sample
            - epistemic_uncertainty_per_sample
            - mean_total_uncertainty
            - mean_aleatoric_uncertainty
            - mean_epistemic_uncertainty
            - epistemic_variance_per_sample
            - mean_epistemic_variance
    """

    preds_mc = np.array(preds_mc)

    if preds_mc.ndim != 3:
        raise ValueError(
            f"preds_mc should have shape (num_mc_samples, N, num_classes), "
            f"but got shape {preds_mc.shape}"
        )

    num_mc_samples, num_points, num_classes = preds_mc.shape

    # -------------------------------------------------
    # Convert logits to probabilities if needed
    # -------------------------------------------------
    if input_type == "logits":
        logits = preds_mc

        # stable softmax
        logits = logits - np.max(logits, axis=2, keepdims=True)
        exp_logits = np.exp(logits)
        probs_mc = exp_logits / np.sum(exp_logits, axis=2, keepdims=True)

    elif input_type == "probs":
        probs_mc = preds_mc

    else:
        raise ValueError("input_type must be either 'logits' or 'probs'.")

    probs_mc = np.clip(probs_mc, eps, 1.0)

    # -------------------------------------------------
    # Mean predictive probability
    # p_bar(y|x) = E_w[p(y|x,w)]
    # -------------------------------------------------
    predictive_probs = np.mean(probs_mc, axis=0)   # shape: (N, C)
    predictive_probs = np.clip(predictive_probs, eps, 1.0)

    # -------------------------------------------------
    # Total uncertainty: entropy of mean prediction
    # H(E[p])
    # -------------------------------------------------
    total_uncertainty = -np.sum(
        predictive_probs * np.log(predictive_probs),
        axis=1
    )   # shape: (N,)

    # -------------------------------------------------
    # Aleatoric uncertainty:
    # E[H(p)]
    # -------------------------------------------------
    entropy_each_mc = -np.sum(
        probs_mc * np.log(probs_mc),
        axis=2
    )   # shape: (num_mc_samples, N)

    aleatoric_uncertainty = np.mean(
        entropy_each_mc,
        axis=0
    )   # shape: (N,)

    # -------------------------------------------------
    # Epistemic uncertainty:
    # H(E[p]) - E[H(p)]
    # This is mutual information / BALD-style uncertainty.
    # -------------------------------------------------
    epistemic_uncertainty = total_uncertainty - aleatoric_uncertainty

    # Numerical safety
    epistemic_uncertainty = np.maximum(epistemic_uncertainty, 0.0)

    # -------------------------------------------------
    # Optional normalization to [0, 1]
    # -------------------------------------------------
    if normalize:
        normalizer = np.log(num_classes)

        total_uncertainty = total_uncertainty / normalizer
        aleatoric_uncertainty = aleatoric_uncertainty / normalizer
        epistemic_uncertainty = epistemic_uncertainty / normalizer

    # -------------------------------------------------
    # Another useful epistemic proxy:
    # variance of class probabilities across MC samples
    # -------------------------------------------------
    epistemic_variance = np.sum(
        np.var(probs_mc, axis=0),
        axis=1
    )   # shape: (N,)

    uncertainty_dict = {
        "predictive_probs": predictive_probs,

        "total_uncertainty_per_sample": total_uncertainty,
        "aleatoric_uncertainty_per_sample": aleatoric_uncertainty,
        "epistemic_uncertainty_per_sample": epistemic_uncertainty,

        "mean_total_uncertainty": float(np.mean(total_uncertainty)),
        "mean_aleatoric_uncertainty": float(np.mean(aleatoric_uncertainty)),
        "mean_epistemic_uncertainty": float(np.mean(epistemic_uncertainty)),

        "epistemic_variance_per_sample": epistemic_variance,
        "mean_epistemic_variance": float(np.mean(epistemic_variance)),
    }

    return uncertainty_dict

In [ ]:
for name, model in models_dict.items():
    base_model = model.eval()

    preds_mc_logits, all_labels = bayesian_predict_with_uncertainty(
        base_model,
        testloader,
        num_samples=num_mc_samples,
        dropconnect_type=name,
        drop_prob=dropout_ratio,
        return_logits=True
    )

    uncertainty_results = compute_classification_uncertainty_from_mc(
        preds_mc_logits,
        input_type="logits",
        normalize=True
    )

    print(f"\nModel: {name.upper()}")
    print(f"Mean Total Uncertainty     : {uncertainty_results['mean_total_uncertainty']:.6f}")
    print(f"Mean Aleatoric Uncertainty : {uncertainty_results['mean_aleatoric_uncertainty']:.6f}")
    print(f"Mean Epistemic Uncertainty : {uncertainty_results['mean_epistemic_uncertainty']:.6f}")
    print(f"Mean Epistemic Variance    : {uncertainty_results['mean_epistemic_variance']:.6f}")

In [ ]:
print("Final models_dict keys:", list(models_dict.keys()))
print("Concrete p:", models_dict["concrete"].get_concrete_dropout_prob())

In [ ]:
### model save for resnet18 only
import os
import json
import torch

# -------------------------------------------------------
# Set the output directory as needed
# -------------------------------------------------------
SAVE_DIR = r"results/fashion-mnist/0.3/vgg16/"

os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(SAVE_DIR, "concrete_models_dict_checkpoint.pt")
METADATA_PATH = os.path.join(SAVE_DIR, "concrete_metadata.json")


def state_dict_to_cpu(model):
    return {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }


def get_model_spec(name, model):
    conv1 = model.backbone.conv1
    fc = model.backbone.fc

    in_channels = conv1.in_channels

    if hasattr(fc, "out_features"):
        num_classes = fc.out_features
    else:
        num_classes = 10

    if name.lower() == "concrete" or getattr(model, "drop_type", None) == "concrete":
        return {
            "class_name": "ConcreteResNet18_FC",
            "drop_type": "concrete",
            "drop_prob": float(model.get_concrete_dropout_prob()),
            "initial_drop_prob": float(getattr(model, "drop_prob", model.get_concrete_dropout_prob())),
            "alpha": float(getattr(model, "alpha", 50)),
            "in_channels": int(in_channels),
            "num_classes": int(num_classes),
            "weight_regularizer": float(getattr(model, "weight_regularizer", 1e-6)),
            "dropout_regularizer": float(getattr(model, "dropout_regularizer", 1e-5)),
            "concrete_temperature": float(getattr(model, "concrete_temperature", 0.1)),
        }

    else:
        return {
            "class_name": "BayesianResNet18_FC",
            "drop_type": getattr(model, "drop_type", None),
            "drop_prob": float(getattr(model, "drop_prob", 0.0)),
            "alpha": float(getattr(model, "alpha", 50)),
            "in_channels": int(in_channels),
            "num_classes": int(num_classes),
        }


def save_models_dict(
    models_dict,
    save_path,
    metadata_path=None,
    dataset_name=None,
    num_epochs=None,
    initial_p=None,
    learned_concrete_p=None
):
    checkpoint = {
        "state_dicts": {},
        "model_specs": {},
        "extra_info": {
            "dataset_name": dataset_name,
            "num_epochs": int(num_epochs) if num_epochs is not None else None,
            "initial_p": float(initial_p) if initial_p is not None else None,
            "learned_concrete_p": float(learned_concrete_p) if learned_concrete_p is not None else None,
        }
    }

    for name, model in models_dict.items():
        model.eval()

        checkpoint["state_dicts"][name] = state_dict_to_cpu(model)
        checkpoint["model_specs"][name] = get_model_spec(name, model)

    torch.save(checkpoint, save_path)

    if metadata_path is not None:
        metadata = {
            "saved_models": list(models_dict.keys()),
            "model_specs": checkpoint["model_specs"],
            "extra_info": checkpoint["extra_info"],
            "checkpoint_file": os.path.basename(save_path),
        }

        with open(metadata_path, "w") as f:
            json.dump(metadata, f, indent=4)

    print(f"Saved checkpoint to:\n{save_path}")

    if metadata_path is not None:
        print(f"Saved metadata to:\n{metadata_path}")


learned_concrete_p = models_dict["concrete"].get_concrete_dropout_prob()

save_models_dict(
    models_dict=models_dict,
    save_path=CHECKPOINT_PATH,
    metadata_path=METADATA_PATH,
    dataset_name=dataset_name,
    num_epochs=num_epochs,
    initial_p=dropout_ratio if "initial_p" in globals() else None,
    learned_concrete_p=learned_concrete_p
)

In [ ]:
### model save for vgg16 and resnet18 generalized
### model save for ResNet18 / VGG16
import os
import json
import torch
import numpy as np

# -------------------------------------------------------
# Set the output directory as needed for each experiment
# -------------------------------------------------------
SAVE_DIR = r"results/fashion-mnist/0.5/vgg16/"

os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(SAVE_DIR, "models_dict_checkpoint.pt")
METADATA_PATH = os.path.join(SAVE_DIR, "metadata.json")


# -------------------------------------------------------
# Utility: move state_dict to CPU before saving
# -------------------------------------------------------
def state_dict_to_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


# -------------------------------------------------------
# Infer input channels for ResNet18 or VGG16
# -------------------------------------------------------
def infer_in_channels(model):
    # ResNet-style model
    # e.g. model.backbone.conv1
    if hasattr(model, "backbone") and hasattr(model.backbone, "conv1"):
        return model.backbone.conv1.in_channels

    # VGG-style model
    # e.g. model.backbone.features[0]
    if hasattr(model, "backbone") and hasattr(model.backbone, "features"):
        first_layer = model.backbone.features[0]
        if hasattr(first_layer, "in_channels"):
            return first_layer.in_channels

    raise ValueError(
        f"Could not infer in_channels for model class: {model.__class__.__name__}"
    )


# -------------------------------------------------------
# Infer number of classes for ResNet18 or VGG16
# -------------------------------------------------------
def infer_num_classes(model):
    # VGG-style custom classifier
    # e.g. model.classifier = BayesLinear / Linear
    if hasattr(model, "classifier"):
        head = model.classifier

        if hasattr(head, "out_features"):
            return head.out_features

        if hasattr(head, "weight_mu"):
            return head.weight_mu.shape[0]

        if hasattr(head, "weight"):
            return head.weight.shape[0]

    # ResNet-style final layer
    # e.g. model.backbone.fc = BayesLinear / Linear
    if hasattr(model, "backbone") and hasattr(model.backbone, "fc"):
        head = model.backbone.fc

        if hasattr(head, "out_features"):
            return head.out_features

        if hasattr(head, "weight_mu"):
            return head.weight_mu.shape[0]

        if hasattr(head, "weight"):
            return head.weight.shape[0]

    raise ValueError(
        f"Could not infer num_classes for model class: {model.__class__.__name__}"
    )


# -------------------------------------------------------
# Build serializable model specification
# -------------------------------------------------------
def get_model_spec(name, model):
    class_name = model.__class__.__name__

    in_channels = infer_in_channels(model)
    num_classes = infer_num_classes(model)

    drop_type = getattr(model, "drop_type", None)

    is_concrete = (
        name.lower() == "concrete"
        or drop_type == "concrete"
        or hasattr(model, "concrete_p_logit")
    )

    if is_concrete:
        return {
            "class_name": class_name,
            "drop_type": "concrete",

            # Initial/drop setting used to construct the model
            "drop_prob": float(getattr(model, "drop_prob", 0.3)),

            # Learned Concrete Dropout probability after training
            "learned_concrete_p": float(model.get_concrete_dropout_prob()),

            "alpha": float(getattr(model, "alpha", 50)),
            "in_channels": int(in_channels),
            "num_classes": int(num_classes),

            "weight_regularizer": float(
                getattr(model, "weight_regularizer", 1e-6)
            ),
            "dropout_regularizer": float(
                getattr(model, "dropout_regularizer", 1e-5)
            ),
            "concrete_temperature": float(
                getattr(model, "concrete_temperature", 0.1)
            ),
        }

    else:
        return {
            "class_name": class_name,
            "drop_type": drop_type,
            "drop_prob": float(getattr(model, "drop_prob", 0.0)),
            "alpha": float(getattr(model, "alpha", 50)),
            "in_channels": int(in_channels),
            "num_classes": int(num_classes),
        }


# -------------------------------------------------------
# Save full models_dict checkpoint
# -------------------------------------------------------
def save_models_dict(
    models_dict,
    save_path,
    metadata_path=None,
    dataset_name=None,
    num_epochs=None,
    initial_p=None
):
    checkpoint = {
        "state_dicts": {},
        "model_specs": {},
        "extra_info": {
            "dataset_name": dataset_name,
            "num_epochs": int(num_epochs) if num_epochs is not None else None,
            "initial_p": float(initial_p) if initial_p is not None else None,
        }
    }

    learned_concrete_p = None

    for name, model in models_dict.items():
        model.eval()

        checkpoint["state_dicts"][name] = state_dict_to_cpu(model)
        checkpoint["model_specs"][name] = get_model_spec(name, model)

        if checkpoint["model_specs"][name]["drop_type"] == "concrete":
            learned_concrete_p = checkpoint["model_specs"][name]["learned_concrete_p"]

    checkpoint["extra_info"]["learned_concrete_p"] = learned_concrete_p

    torch.save(checkpoint, save_path)

    if metadata_path is not None:
        metadata = {
            "saved_models": list(models_dict.keys()),
            "model_specs": checkpoint["model_specs"],
            "extra_info": checkpoint["extra_info"],
            "checkpoint_file": os.path.basename(save_path),
        }

        with open(metadata_path, "w") as f:
            json.dump(metadata, f, indent=4)

    print(f"Saved checkpoint to:\n{save_path}")

    if metadata_path is not None:
        print(f"Saved metadata to:\n{metadata_path}")

    print("\nSaved models:")
    for name in checkpoint["state_dicts"]:
        print(f"  - {name}: {checkpoint['model_specs'][name]['class_name']}")

    return checkpoint


# -------------------------------------------------------
# Actually save current models_dict
# -------------------------------------------------------
checkpoint = save_models_dict(
    models_dict=models_dict,
    save_path=CHECKPOINT_PATH,
    metadata_path=METADATA_PATH,
    dataset_name=dataset_name if "dataset_name" in globals() else None,
    num_epochs=num_epochs if "num_epochs" in globals() else None,
    initial_p=dropout_ratio if "dropout_ratio" in globals() else None
)

In [ ]:
##### Loading models


# ============================================================
# Reload saved VGG16 models from checkpoint
# ============================================================

import os
import torch
import numpy as np

# ------------------------------------------------------------
# Set saved location
# ------------------------------------------------------------
SAVE_DIR = r"results/fashion-mnist/0.3/vgg16/"

CHECKPOINT_PATH = os.path.join(SAVE_DIR, "models_dict_checkpoint.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading checkpoint from:")
print(CHECKPOINT_PATH)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

saved_state_dicts = checkpoint["state_dicts"]
saved_model_specs = checkpoint["model_specs"]

print("Saved models found:")
print(list(saved_state_dicts.keys()))


# ============================================================
# Helper: rebuild model from saved spec
# ============================================================

def build_model_from_saved_spec(saved_spec):
    class_name = saved_spec["class_name"]

    if class_name == "BayesianVGG16_FC":
        model = BayesianVGG16_FC(
            drop_type=saved_spec.get("drop_type", None),
            drop_prob=saved_spec.get("drop_prob", 0.3),
            alpha=saved_spec.get("alpha", 50),
            in_channels=saved_spec.get("in_channels", 1),
            num_classes=saved_spec.get("num_classes", 10)
        )

    elif class_name == "ConcreteVGG16_FC":
        model = ConcreteVGG16_FC(
            drop_prob=saved_spec.get("drop_prob", 0.3),
            alpha=saved_spec.get("alpha", 50),
            in_channels=saved_spec.get("in_channels", 1),
            num_classes=saved_spec.get("num_classes", 10),
            weight_regularizer=saved_spec.get("weight_regularizer", 1e-6),
            dropout_regularizer=saved_spec.get("dropout_regularizer", 1e-5),
            concrete_temperature=saved_spec.get("concrete_temperature", 0.1)
        )

    else:
        raise ValueError(f"Unknown class_name in checkpoint: {class_name}")

    return model


# ============================================================
# Load all models into models_dict
# ============================================================

models_dict = {}

for name in saved_state_dicts.keys():

    print("\nLoading model:", name)

    saved_spec = saved_model_specs[name]

    model = build_model_from_saved_spec(saved_spec)

    model.load_state_dict(
        saved_state_dicts[name],
        strict=True
    )

    model.to(device)
    model.eval()

    models_dict[name] = model

    print(f"Loaded {name}: {saved_spec['class_name']}")

    if getattr(model, "drop_type", None) == "concrete":
        print("Learned concrete p:", model.get_concrete_dropout_prob())


print("\nAll models loaded successfully.")
print(models_dict.keys())

In [ ]:
#### plots of training loss and val F1

import matplotlib.pyplot as plt

def plot_training_curves(train_losses, val_losses, val_f1s, save_path=None):
    # Training loss
    plt.figure(figsize=(7, 5))
    for name, values in train_losses.items():
        plt.plot(range(1, len(values) + 1), values, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Training loss")
    #plt.title("Training loss across methods")
    plt.legend()
    plt.grid(True, alpha=0.3)
    if save_path is not None:
        plt.savefig(save_path + "_train_loss.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Validation loss
    plt.figure(figsize=(7, 5))
    for name, values in val_losses.items():
        plt.plot(range(1, len(values) + 1), values, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Validation loss")
    #plt.title("Validation loss across methods")
    plt.legend()
    plt.grid(True, alpha=0.3)
    if save_path is not None:
        plt.savefig(save_path + "_val_loss.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Validation F1
    plt.figure(figsize=(7, 5))
    for name, values in val_f1s.items():
        plt.plot(range(1, len(values) + 1), values, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Validation macro-F1")
    #plt.title("Validation F1 across methods")
    plt.legend()
    plt.grid(True, alpha=0.3)
    if save_path is not None:
        plt.savefig(save_path + "_val_f1.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
plot_training_curves(
    train_losses=train_losses,
    val_losses=val_losses,
    val_f1s=val_f1s,
    save_path="fashionmnist_vgg16_p03"
)

In [ ]:
#### ECDF plots

# ============================================================
# MC evaluation + ECDF-ready metric storage
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

num_mc_samples = 50
eps = 1e-10

pretty_names = {
    "plain": "Plain BNN",
    "dropout": "Dropout",
    "dropconnect": "DropConnect",
    "entropy": "Sign-Entropy",
    "concrete": "Concrete Dropout",
}

metrics = {
    name: {
        "F1": [],
        "NLL": [],
        "Brier": [],
        "ECE": [],
        "AECE": []
    }
    for name in models_dict.keys()
}

uncertainty_pointwise = {}
uncertainty_summary_rows = []


def softmax_numpy(logits, axis=-1):
    logits = logits - np.max(logits, axis=axis, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=axis, keepdims=True)


def compute_uncertainty_from_mc_probs(probs_mc, eps=1e-10):
    """
    probs_mc: shape (T, N, C)

    Returns:
        total_entropy: shape (N,)
        aleatoric_uncertainty: shape (N,)
        epistemic_uncertainty: shape (N,)
    """

    mean_probs = np.mean(probs_mc, axis=0)  # (N, C)

    total_entropy = -np.sum(
        mean_probs * np.log(mean_probs + eps),
        axis=1
    )

    entropy_each_mc = -np.sum(
        probs_mc * np.log(probs_mc + eps),
        axis=2
    )  # (T, N)

    aleatoric_uncertainty = np.mean(entropy_each_mc, axis=0)  # (N,)

    epistemic_uncertainty = total_entropy - aleatoric_uncertainty
    epistemic_uncertainty = np.maximum(epistemic_uncertainty, 0.0)

    return total_entropy, aleatoric_uncertainty, epistemic_uncertainty


for name, model in models_dict.items():

    print("\n" + "=" * 80)
    print(f"MC evaluation for: {name}")
    print("=" * 80)

    base_model = model.eval()

    preds_mc_logits, all_labels = bayesian_predict_with_uncertainty(
        base_model,
        testloader,
        num_samples=num_mc_samples,
        dropconnect_type=name,
        drop_prob=dropout_ratio,
        return_logits=True
    )

    # Shape: (T, N, C)
    preds_mc_probs = softmax_numpy(preds_mc_logits, axis=2)

    # --------------------------------------------------------
    # Pointwise uncertainty for ECDF
    # --------------------------------------------------------
    total_u, aleatoric_u, epistemic_u = compute_uncertainty_from_mc_probs(
        preds_mc_probs,
        eps=eps
    )

    uncertainty_pointwise[name] = {
        "Total_U": total_u,
        "Aleatoric_U": aleatoric_u,
        "Epistemic_U": epistemic_u,
    }

    uncertainty_summary_rows.append({
        "Method": name,
        "Mean_Total_U": np.mean(total_u),
        "Mean_Aleatoric_U": np.mean(aleatoric_u),
        "Mean_Epistemic_U": np.mean(epistemic_u),
    })

    print(f"Mean Total Uncertainty     : {np.mean(total_u):.6f}")
    print(f"Mean Aleatoric Uncertainty : {np.mean(aleatoric_u):.6f}")
    print(f"Mean Epistemic Uncertainty : {np.mean(epistemic_u):.6f}")

    # --------------------------------------------------------
    # Per-MC-run metrics for ECDF
    # --------------------------------------------------------
    for mc_idx in range(num_mc_samples):

        logits = preds_mc_logits[mc_idx]  # (N, C)
        probs = softmax_numpy(logits, axis=1)

        y_pred = np.argmax(probs, axis=1)

        f1 = f1_score(all_labels, y_pred, average="macro")

        nll = -np.mean(
            np.log(probs[np.arange(len(all_labels)), all_labels] + eps)
        )

        true_labels_onehot = np.eye(probs.shape[1])[all_labels]

        brier = np.mean(
            np.sum((probs - true_labels_onehot) ** 2, axis=1)
        )

        ece = compute_ece(
            y_true=all_labels,
            y_prob=probs
        )

        aece = compute_aece(
            y_true=all_labels,
            y_prob=probs
        )

        metrics[name]["F1"].append(f1)
        metrics[name]["NLL"].append(nll)
        metrics[name]["Brier"].append(brier)
        metrics[name]["ECE"].append(ece)
        metrics[name]["AECE"].append(aece)

    print(f"Finished MC sampling for {name}")


uncertainty_summary_df = pd.DataFrame(uncertainty_summary_rows)
uncertainty_summary_df

In [ ]:
# ============================================================
# ECDF plots for F1, NLL, Brier, AECE, Aleatoric U, Epistemic U
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt


def ecdf(values):
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    values = np.sort(values)

    if len(values) == 0:
        return values, values

    y = np.arange(1, len(values) + 1) / len(values)
    return values, y


def plot_metric_ecdf(ax, metrics, metric_name, pretty_names=None):
    for name in metrics.keys():
        if metric_name not in metrics[name]:
            continue

        values = metrics[name][metric_name]

        if len(values) == 0:
            continue

        x, y = ecdf(values)

        label = pretty_names.get(name, name) if pretty_names is not None else name
        ax.plot(x, y, linewidth=2, label=label)

    ax.set_title(metric_name)
    ax.set_xlabel(metric_name)
    ax.set_ylabel("ECDF")
    ax.grid(True, alpha=0.3)


def plot_uncertainty_ecdf(ax, uncertainty_pointwise, uncertainty_name, pretty_names=None):
    for name in uncertainty_pointwise.keys():
        if uncertainty_name not in uncertainty_pointwise[name]:
            continue

        values = uncertainty_pointwise[name][uncertainty_name]

        if len(values) == 0:
            continue

        x, y = ecdf(values)

        label = pretty_names.get(name, name) if pretty_names is not None else name
        ax.plot(x, y, linewidth=2, label=label)

    ax.set_title(uncertainty_name)
    ax.set_xlabel(uncertainty_name)
    ax.set_ylabel("ECDF")
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(2, 3, figsize=(18, 10))

plot_metric_ecdf(
    axes[0, 0],
    metrics,
    "F1",
    pretty_names=pretty_names
)

plot_metric_ecdf(
    axes[0, 1],
    metrics,
    "NLL",
    pretty_names=pretty_names
)

plot_metric_ecdf(
    axes[0, 2],
    metrics,
    "Brier",
    pretty_names=pretty_names
)

plot_metric_ecdf(
    axes[1, 0],
    metrics,
    "AECE",
    pretty_names=pretty_names
)

plot_uncertainty_ecdf(
    axes[1, 1],
    uncertainty_pointwise,
    "Aleatoric_U",
    pretty_names=pretty_names
)

plot_uncertainty_ecdf(
    axes[1, 2],
    uncertainty_pointwise,
    "Epistemic_U",
    pretty_names=pretty_names
)

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=len(labels),
    frameon=True,
    fontsize=12
)

fig.suptitle(
    "ECDF comparison across methods",
    fontsize=18
)

plt.tight_layout(rect=[0, 0.08, 1, 0.95])

ecdf_save_path = os.path.join(
    "",
    "fashionmnist_vgg16_p03_ecdf_metrics_uncertainty.png"
)

plt.savefig(
    ecdf_save_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved ECDF figure to:")
print(ecdf_save_path)

In [ ]:
##### ECDF plots over test samples and not MC runs

# ============================================================
# Better ECDF plots: pointwise NLL, Brier, Aleatoric U, Epistemic U
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

eps = 1e-10
NORMALIZE_ENTROPY = False   # keep False if you want to match your current uncertainty table
                            # set True if you want uncertainty in [0, 1]

pretty_names = {
    "plain": "Plain BNN",
    "dropout": "Dropout",
    "dropconnect": "DropConnect",
    "entropy": "Sign-Entropy",
    "concrete": "Concrete Dropout",
}


def softmax_numpy(logits, axis=-1):
    logits = logits - np.max(logits, axis=axis, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=axis, keepdims=True)


def ecdf(values):
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    values = np.sort(values)

    y = np.arange(1, len(values) + 1) / len(values)
    return values, y


def compute_pointwise_classification_quantities(probs_mc, y_true, eps=1e-10, normalize_entropy=False):
    """
    probs_mc: shape (T, N, C)
    y_true  : shape (N,)

    Returns pointwise quantities of shape (N,).
    """

    T, N, C = probs_mc.shape

    mean_probs = np.mean(probs_mc, axis=0)  # (N, C)

    # Per-sample NLL using MC-averaged predictive probability
    pointwise_nll = -np.log(
        mean_probs[np.arange(N), y_true] + eps
    )

    # Per-sample Brier score using MC-averaged predictive probability
    y_onehot = np.eye(C)[y_true]

    pointwise_brier = np.sum(
        (mean_probs - y_onehot) ** 2,
        axis=1
    )

    # Predictive entropy: H[E[p(y|x,w)]]
    total_entropy = -np.sum(
        mean_probs * np.log(mean_probs + eps),
        axis=1
    )

    # Expected entropy: E[H[p(y|x,w)]]
    entropy_each_mc = -np.sum(
        probs_mc * np.log(probs_mc + eps),
        axis=2
    )  # (T, N)

    aleatoric_u = np.mean(entropy_each_mc, axis=0)

    # Mutual information: epistemic uncertainty
    epistemic_u = total_entropy - aleatoric_u
    epistemic_u = np.maximum(epistemic_u, 0.0)

    if normalize_entropy:
        normalizer = np.log(C)
        total_entropy = total_entropy / normalizer
        aleatoric_u = aleatoric_u / normalizer
        epistemic_u = epistemic_u / normalizer

    return {
        "Pointwise NLL": pointwise_nll,
        "Pointwise Brier": pointwise_brier,
        "Aleatoric U": aleatoric_u,
        "Epistemic U": epistemic_u,
    }


# ------------------------------------------------------------
# Collect pointwise quantities for each method
# ------------------------------------------------------------

pointwise_results = {}

for name, model in models_dict.items():

    print("\n" + "=" * 80)
    print("Pointwise ECDF quantities for:", name)
    print("=" * 80)

    model.eval()

    preds_mc_logits, all_labels = bayesian_predict_with_uncertainty(
        model,
        testloader,
        num_samples=num_mc_samples,
        dropconnect_type=name,
        drop_prob=dropout_ratio,
        return_logits=True
    )

    probs_mc = softmax_numpy(preds_mc_logits, axis=2)

    pointwise_results[name] = compute_pointwise_classification_quantities(
        probs_mc=probs_mc,
        y_true=all_labels,
        eps=eps,
        normalize_entropy=NORMALIZE_ENTROPY
    )

In [ ]:
# ============================================================
# Paired sample-wise scatter:
# Sign-Entropy vs each baseline for Pointwise NLL and Brier
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt


pretty_names = {
    "plain": "Plain",
    "dropout": "Dropout",
    "dropconnect": "DropConnect",
    "entropy": "Proposed",
    "concrete": "Concrete Dropout",
}

baseline_methods = [
    "plain",
    "dropout",
    "dropconnect",
    "concrete"
]

metrics_to_plot = [
    "Pointwise NLL",
    "Pointwise Brier"
]


def clean_pair(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    mask = np.isfinite(x) & np.isfinite(y)

    return x[mask], y[mask]


def plot_entropy_vs_baseline_scatter(ax, baseline_name, metric_name):
    baseline_values = pointwise_results[baseline_name][metric_name]
    entropy_values = pointwise_results["entropy"][metric_name]

    x, y = clean_pair(baseline_values, entropy_values)

    # For readability, zoom to central region.
    all_vals = np.concatenate([x, y])
    axis_min = np.percentile(all_vals, 0.5)
    axis_max = np.percentile(all_vals, 97.5)

    # Scatter: each point is one test sample
    ax.scatter(
        x,
        y,
        s=14,
        alpha=0.18,
        rasterized=True
    )

    # Diagonal reference line
    ax.plot(
        [axis_min, axis_max],
        [axis_min, axis_max],
        linestyle="--",
        linewidth=3,
        color="red"
    )

    ax.set_xlim(axis_min, axis_max)
    ax.set_ylim(axis_min, axis_max)

    # Proposed is better when y < x
    entropy_better = np.mean(y < x)
    entropy_equal_or_better = np.mean(y <= x)
    median_improvement = np.median(x - y)

    # --------------------------------------------------------
    # Short clean title
    # --------------------------------------------------------
    ax.set_title(
        f"{pretty_names[baseline_name]}",
        fontsize=24,
        pad=14
    )

    # --------------------------------------------------------
    # Compact annotation inside plot
    # --------------------------------------------------------
    ax.text(
        0.04,
        0.94,
        f"{100 * entropy_better:.1f}% below\n"
        f"$\\Delta_{{med}}$={median_improvement:.4f}",
        transform=ax.transAxes,
        fontsize=25,
        verticalalignment="top",
        horizontalalignment="left",
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor="black",
            alpha=0.85
        )
    )

    ax.set_xlabel(
        f"{pretty_names[baseline_name]}",
        fontsize=30,
        labelpad=12
    )

    ax.set_ylabel(
        "Proposed",
        fontsize=30,
        labelpad=12
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=25,
        width=1.8,
        length=7
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1.8)

    ax.grid(True, alpha=0.3)

    return {
        "Baseline": baseline_name,
        "Metric": metric_name,
        "SE_better_fraction": entropy_better,
        "SE_equal_or_better_fraction": entropy_equal_or_better,
        "Median_improvement": median_improvement,
        "Mean_improvement": np.mean(x - y)
    }

summary_rows = []

for metric_name in metrics_to_plot:

    fig, axes = plt.subplots(1, 4, figsize=(30, 7.5))

    for ax, baseline_name in zip(axes, baseline_methods):
        row = plot_entropy_vs_baseline_scatter(
            ax=ax,
            baseline_name=baseline_name,
            metric_name=metric_name
        )
        summary_rows.append(row)

    #fig.suptitle(f"Sample-wise paired comparison for {metric_name}",fontsize=18)

    plt.tight_layout(rect=[0, 0, 1, 0.90])

    metric_safe_name = metric_name.lower().replace(" ", "_")

    save_path = os.path.join(
        "",
        f"scatterplot_fashionmnist_vgg16_p03_paired_entropy_vs_baselines_{metric_safe_name}.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved to:")
    print(save_path)


paired_scatter_summary_df = pd.DataFrame(summary_rows)
paired_scatter_summary_df